# Regression Assignment Pipeline
### Decision Tree Regression vs. XGBoost Regression
**Datasets:** `RG-Credit.csv`, `RG-Wage.csv`

This notebook implements the full pipeline required by the assignment:
sequential (non-shuffled) train/validation/test splitting, Decision Tree and
XGBoost regression, validation-only hyperparameter tuning, a **manually
implemented** Mean Squared Error metric (no `sklearn.metrics` used anywhere),
and a final results comparison.

## Section 1: Import Libraries

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Section 9 (defined early so it can be reused throughout)
### Manual MSE Implementation
No library metric function (e.g. `sklearn.metrics.mean_squared_error`) is used anywhere
in this notebook. MSE is implemented directly with NumPy according to its standard
definition:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

Steps: (1) compute the residual for every sample, (2) square each residual,
(3) sum the squared residuals, (4) divide by $n$.

In [2]:
def manual_mse(y_true, y_pred):
    """
    Mean Squared Error, implemented manually with NumPy.
    MSE = (1/n) * sum((y_true - y_pred)^2)
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    n = y_true.shape[0]
    residuals = y_true - y_pred          # step 1
    squared_residuals = residuals ** 2   # step 2
    sum_squared = np.sum(squared_residuals)  # step 3
    mse = sum_squared / n                # step 4
    return float(mse)

## Section 2: Load Dataset

In [3]:
def load_credit(path):
    df = pd.read_csv(path)
    return df, "Balance"


def load_wage(path):
    df = pd.read_csv(path)
    # 'logwage' is a deterministic transform of the target 'wage' -> leakage, must drop.
    # 'region' is constant (single category) in this file -> drop, it carries no information.
    df = df.drop(columns=["logwage", "region"])
    return df, "wage"

## Section 3: Data Inspection

In [4]:
def inspect(df, name):
    report = {
        "dataset": name,
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "columns": list(df.columns),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        "missing_values": {c: int(v) for c, v in df.isna().sum().items()},
    }
    return report


def encode_features(df, target):
    """
    One-hot encode categorical predictors; leave the target untouched.
    """
    X = df.drop(columns=[target]).copy()
    y = df[target].copy().reset_index(drop=True)
    cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    X = X.reset_index(drop=True)
    return X, y

## Section 4: Sequential Train / Validation / Test Split
No shuffling -- strict sequential 70% / 15% / 15% split, implemented manually
with NumPy index arithmetic (not `sklearn.model_selection.train_test_split`,
which shuffles by default).

In [5]:
def sequential_split(X, y):
    n = len(X)
    train_end = int(np.floor(0.70 * n))
    val_end = int(np.floor(0.85 * n))

    X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
    X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
    X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

## Section 5 & 6: Decision Tree Implementation + Hyperparameter Tuning
Tuned hyperparameters: `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion`.
An exhaustive grid search (160 combinations) is performed; every combination is trained
on the training set and scored on the validation set with `manual_mse`.

In [6]:
DT_GRID = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["squared_error", "friedman_mse"],
}


def tune_decision_tree(X_train, y_train, X_val, y_val, grid=DT_GRID):
    results = []
    best = {"mse": np.inf, "params": None, "model": None}
    for depth in grid["max_depth"]:
        for mss in grid["min_samples_split"]:
            for msl in grid["min_samples_leaf"]:
                for crit in grid["criterion"]:
                    params = dict(max_depth=depth, min_samples_split=mss,
                                  min_samples_leaf=msl, criterion=crit,
                                  random_state=RANDOM_STATE)
                    model = DecisionTreeRegressor(**params)
                    model.fit(X_train, y_train)
                    val_pred = model.predict(X_val)
                    val_mse = manual_mse(y_val, val_pred)
                    results.append({**params, "val_mse": val_mse})
                    if val_mse < best["mse"]:
                        best = {"mse": val_mse, "params": params, "model": model}
    results_df = pd.DataFrame(results).sort_values("val_mse").reset_index(drop=True)
    return best, results_df

## Section 7 & 8: XGBoost Implementation + Hyperparameter Tuning
Tuned hyperparameters: `max_depth`, `learning_rate`, `n_estimators`, `subsample`,
`colsample_bytree`, `min_child_weight`, `gamma`. The full grid has 288 combinations;
a fixed-seed **random search** over 60 unique combinations is used to keep runtime
tractable while still exercising every listed hyperparameter.

In [ ]:
XGB_GRID = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}

def tune_xgboost(X_train, y_train, X_val, y_val, grid=XGB_GRID, n_random=60, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    keys = list(grid.keys())
    all_combos = []
    for _ in range(n_random * 3):  # oversample then dedupe
        combo = {k: grid[k][rng.randint(len(grid[k]))] for k in keys}
        all_combos.append(tuple(sorted(combo.items())))
    unique_combos = list(dict.fromkeys(all_combos))[:n_random]

    results = []
    best = {"mse": np.inf, "params": None, "model": None}
    for combo in unique_combos:
        params = dict(combo)
        model = XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=4,
            verbosity=0,
            **params
        )
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        val_mse = manual_mse(y_val, val_pred)
        results.append({**params, "val_mse": val_mse})
        if val_mse < best["mse"]:
            best = {"mse": val_mse, "params": params, "model": model}
    results_df = pd.DataFrame(results).sort_values("val_mse").reset_index(drop=True)
    return best, results_df

## Section 10 & 11: Final Evaluation + Results Comparison

In [8]:
def run_pipeline(path, loader, dataset_name):
    sep = "=" * 70
    print(f"\n{sep}\nDATASET: {dataset_name}\n{sep}")
    df, target = loader(path)
    insp = inspect(df, dataset_name)
    print(json.dumps(insp, indent=2))

    X, y = encode_features(df, target)
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = sequential_split(X, y)
    print(f"Train size: {len(X_train)}  Val size: {len(X_val)}  Test size: {len(X_test)}")

    # ---- Decision Tree ----
    dt_best, dt_results = tune_decision_tree(X_train, y_train, X_val, y_val)
    dt_model = dt_best["model"]
    dt_test_pred = dt_model.predict(X_test)
    dt_test_mse = manual_mse(y_test, dt_test_pred)
    print("\nBest Decision Tree params:", dt_best["params"])
    print("Decision Tree Validation MSE:", dt_best["mse"])
    print("Decision Tree Test MSE:", dt_test_mse)

    # ---- XGBoost ----
    xgb_best, xgb_results = tune_xgboost(X_train, y_train, X_val, y_val)
    xgb_model = xgb_best["model"]
    xgb_test_pred = xgb_model.predict(X_test)
    xgb_test_mse = manual_mse(y_test, xgb_test_pred)
    print("\nBest XGBoost params:", xgb_best["params"])
    print("XGBoost Validation MSE:", xgb_best["mse"])
    print("XGBoost Test MSE:", xgb_test_mse)

    return {
        "dataset_name": dataset_name,
        "inspection": insp,
        "n_features": X.shape[1],
        "feature_names": list(X.columns),
        "split_sizes": {"train": len(X_train), "val": len(X_val), "test": len(X_test)},
        "dt_best_params": dt_best["params"],
        "dt_val_mse": dt_best["mse"],
        "dt_test_mse": dt_test_mse,
        "dt_top5": dt_results.head(5).to_dict(orient="records"),
        "xgb_best_params": xgb_best["params"],
        "xgb_val_mse": xgb_best["mse"],
        "xgb_test_mse": xgb_test_mse,
        "xgb_top5": xgb_results.head(5).to_dict(orient="records"),
        "y_stats": {
            "train_mean": float(y_train.mean()), "train_std": float(y_train.std()),
            "test_mean": float(y_test.mean()), "test_std": float(y_test.std()),
        },
    }

### Run on RG-Credit (target: `Balance`)

In [9]:
results_credit = run_pipeline("RG-Credit.csv", load_credit, "RG-Credit (Balance)")


DATASET: RG-Credit (Balance)
{
  "dataset": "RG-Credit (Balance)",
  "n_rows": 400,
  "n_cols": 11,
  "columns": [
    "Income",
    "Limit",
    "Rating",
    "Cards",
    "Age",
    "Education",
    "Own",
    "Student",
    "Married",
    "Region",
    "Balance"
  ],
  "dtypes": {
    "Income": "float64",
    "Limit": "int64",
    "Rating": "int64",
    "Cards": "int64",
    "Age": "int64",
    "Education": "int64",
    "Own": "str",
    "Student": "str",
    "Married": "str",
    "Region": "str",
    "Balance": "int64"
  },
  "missing_values": {
    "Income": 0,
    "Limit": 0,
    "Rating": 0,
    "Cards": 0,
    "Age": 0,
    "Education": 0,
    "Own": 0,
    "Student": 0,
    "Married": 0,
    "Region": 0,
    "Balance": 0
  }
}
Train size: 280  Val size: 60  Test size: 60



Best Decision Tree params: {'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'criterion': 'squared_error', 'random_state': 42}
Decision Tree Validation MSE: 14858.318205761318
Decision Tree Test MSE: 19674.27413088939



Best XGBoost params: {'colsample_bytree': 1.0, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
XGBoost Validation MSE: 7694.6003163823025
XGBoost Test MSE: 3949.2020105741976


### Run on RG-Wage (target: `wage`)

In [10]:
results_wage = run_pipeline("RG-Wage.csv", load_wage, "RG-Wage (wage)")


DATASET: RG-Wage (wage)
{
  "dataset": "RG-Wage (wage)",
  "n_rows": 3000,
  "n_cols": 9,
  "columns": [
    "year",
    "age",
    "maritl",
    "race",
    "education",
    "jobclass",
    "health",
    "health_ins",
    "wage"
  ],
  "dtypes": {
    "year": "int64",
    "age": "int64",
    "maritl": "str",
    "race": "str",
    "education": "str",
    "jobclass": "str",
    "health": "str",
    "health_ins": "str",
    "wage": "float64"
  },
  "missing_values": {
    "year": 0,
    "age": 0,
    "maritl": 0,
    "race": 0,
    "education": 0,
    "jobclass": 0,
    "health": 0,
    "health_ins": 0,
    "wage": 0
  }
}
Train size: 2100  Val size: 450  Test size: 450



Best Decision Tree params: {'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 10, 'criterion': 'squared_error', 'random_state': 42}
Decision Tree Validation MSE: 1323.6681010536875
Decision Tree Test MSE: 1252.9386435436943



Best XGBoost params: {'colsample_bytree': 1.0, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
XGBoost Validation MSE: 1226.1794731834316
XGBoost Test MSE: 1228.4438852117557


### Combined Results Comparison Table

In [11]:
comparison_rows = []
for key, res in [("RG-Credit", results_credit), ("RG-Wage", results_wage)]:
    comparison_rows.append({
        "Dataset": key, "Model": "Decision Tree",
        "Validation MSE": res["dt_val_mse"], "Test MSE": res["dt_test_mse"],
    })
    comparison_rows.append({
        "Dataset": key, "Model": "XGBoost",
        "Validation MSE": res["xgb_val_mse"], "Test MSE": res["xgb_test_mse"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Dataset,Model,Validation MSE,Test MSE
0,RG-Credit,Decision Tree,14858.318206,19674.274131
1,RG-Credit,XGBoost,7694.600316,3949.202011
2,RG-Wage,Decision Tree,1323.668101,1252.938644
3,RG-Wage,XGBoost,1226.179473,1228.443885


In [12]:
all_results = {"credit": results_credit, "wage": results_wage}
with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print("Saved results.json")

Saved results.json
